# exp047 BC + iNat + AnuraSet Extract (Kaggle)

**Purpose**: BC2026 234 species を **過去 BC + iNat + AnuraSet** から抽出、Phase 2 pretrain 用に統合。

**Inputs** (7 sources):
| Source | Type | 期待 BC2026 coverage |
|---|---|---|
| birdclef-2021 | competition | ~34 Aves species |
| birdclef-2022 | competition | ~5 species |
| birdclef-2023 | competition | ~1 species |
| birdclef-2024 | competition | ~1 species |
| birdclef-2025 | competition | ~42 species |
| shadowdude/train-recordings (iNat) | dataset | ~31 species (含 non-Aves) |
| bengtlueers/anuraset-v2-raw | dataset | 17 Amphibia |

**Filter**: BC2026 234 species (scientific_name または primary_label match)

**Output**:
- `/kaggle/working/extracted/audio/{source}/{primary_label}/*.ogg|wav|mp3`
- `metadata.csv` (filename, primary_label, source, scientific_name, class_name)
- Upload → `maekeso/birdclef2026-exp047-bc-inat-anura-extracted`

**Expected**:
- ~80-150 unique BC2026 species covered (union over sources)
- ~15-25 GB after per-species cap

**Safety**:
- Per-species cap (200 file/species)
- Storage cap 18 GB
- Per-source failure に強い (defensive)

**Note**: 過去 BC competition は **rule accept 必要** (各 URL で "Late Submission" 承諾)


In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import os, sys, json, time, re, shutil
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True, parents=True)
AUDIO_DIR = OUT_DIR / "audio"
AUDIO_DIR.mkdir(exist_ok=True)

# Caps
MAX_PER_SPECIES = 200
STORAGE_CAP_GB = 18.0

START_T = time.time()
print(f"Output: {OUT_DIR}")


In [ ]:
# ============================================================
# Cell 2: BC2026 species list (234) with inat_taxon_id
# ============================================================
# tuple: (scientific_name, primary_label, class_name, inat_taxon_id)
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta', 1161364),
    ('Caiman yacare', '116570', 'Reptilia', 116570),
    ('Leptodactylus luctator', '1176823', 'Amphibia', 1176823),
    ('Adenomera guarani', '1491113', 'Amphibia', 1491113),
    ('Lysapsus limellum', '1595929', 'Amphibia', 1595929),
    ('Equus caballus', '209233', 'Mammalia', 209233),
    ('Leptodactylus syphax', '22930', 'Amphibia', 22930),
    ('Leptodactylus mystacinus', '22956', 'Amphibia', 22956),
    ('Leptodactylus podicipinus', '22961', 'Amphibia', 22961),
    ('Leptodactylus elenae', '22967', 'Amphibia', 22967),
    ('Leptodactylus fuscus', '22973', 'Amphibia', 22973),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia', 22983),
    ('Leptodactylus petersii', '22985', 'Amphibia', 22985),
    ('Physalaemus centralis', '23150', 'Amphibia', 23150),
    ('Physalaemus albifrons', '23154', 'Amphibia', 23154),
    ('Physalaemus albonotatus', '23158', 'Amphibia', 23158),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia', 23176),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia', 23724),
    ('Scinax nasicus', '24279', 'Amphibia', 24279),
    ('Scinax fuscovarius', '24285', 'Amphibia', 24285),
    ('Scinax fuscomarginatus', '24287', 'Amphibia', 24287),
    ('Scinax acuminatus', '24321', 'Amphibia', 24321),
    ('Quesada gigas', '244024', 'Insecta', 244024),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia', 25073),
    ('Elachistocleis bicolor', '25092', 'Amphibia', 25092),
    ('Dermatonotus muelleri', '25214', 'Amphibia', 25214),
    ('Physalaemus biligonigerus', '326272', 'Amphibia', 326272),
    ('Panthera onca', '41970', 'Mammalia', 41970),
    ('Alouatta caraya', '43435', 'Mammalia', 43435),
    ('Canis familiaris', '47144', 'Mammalia', 47144),
    ('Insect son01', '47158son01', 'Insecta', 47158),
    ('Insect son02', '47158son02', 'Insecta', 47158),
    ('Insect son03', '47158son03', 'Insecta', 47158),
    ('Insect son04', '47158son04', 'Insecta', 47158),
    ('Insect son05', '47158son05', 'Insecta', 47158),
    ('Insect son06', '47158son06', 'Insecta', 47158),
    ('Insect son07', '47158son07', 'Insecta', 47158),
    ('Insect son08', '47158son08', 'Insecta', 47158),
    ('Insect son09', '47158son09', 'Insecta', 47158),
    ('Insect son10', '47158son10', 'Insecta', 47158),
    ('Insect son11', '47158son11', 'Insecta', 47158),
    ('Insect son12', '47158son12', 'Insecta', 47158),
    ('Insect son13', '47158son13', 'Insecta', 47158),
    ('Insect son14', '47158son14', 'Insecta', 47158),
    ('Insect son15', '47158son15', 'Insecta', 47158),
    ('Insect son16', '47158son16', 'Insecta', 47158),
    ('Insect son17', '47158son17', 'Insecta', 47158),
    ('Insect son18', '47158son18', 'Insecta', 47158),
    ('Insect son19', '47158son19', 'Insecta', 47158),
    ('Insect son20', '47158son20', 'Insecta', 47158),
    ('Insect son21', '47158son21', 'Insecta', 47158),
    ('Insect son22', '47158son22', 'Insecta', 47158),
    ('Insect son23', '47158son23', 'Insecta', 47158),
    ('Insect son24', '47158son24', 'Insecta', 47158),
    ('Insect son25', '47158son25', 'Insecta', 47158),
    ('Physalaemus nattereri', '476521', 'Amphibia', 476521),
    ('Sapajus cay', '516975', 'Mammalia', 516975),
    ('Pithecopus azureus', '517063', 'Amphibia', 517063),
    ('Boana lundii', '555123', 'Amphibia', 555123),
    ('Boana punctata', '555145', 'Amphibia', 555145),
    ('Boana raniceps', '555146', 'Amphibia', 555146),
    ('Ameerega picta', '64898', 'Amphibia', 64898),
    ('Dendropsophus minutus', '65377', 'Amphibia', 65377),
    ('Dendropsophus nanus', '65380', 'Amphibia', 65380),
    ('Pseudis platensis', '66971', 'Amphibia', 66971),
    ('Rhinella diptycha', '67107', 'Amphibia', 67107),
    ('Trachycephalus typhonius', '67252', 'Amphibia', 67252),
    ('Leptodactylus macrosternum', '70711', 'Amphibia', 70711),
    ('Plecturocebus pallescens', '738183', 'Mammalia', 738183),
    ('Bos taurus', '74113', 'Mammalia', 74113),
    ('Mico melanurus', '74580', 'Mammalia', 74580),
    ('Prionacris erosa', '760266', 'Insecta', 760266),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves', 17431),
    ('Mustelirallus albicollis', 'astcra1', 'Aves', 508907),
    ('Crax fasciolata', 'bafcur1', 'Aves', 2046),
    ('Micrastur ruficollis', 'baffal1', 'Aves', 4699),
    ('Coereba flaveola', 'banana', 'Aves', 10199),
    ('Thamnophilus doliatus', 'barant1', 'Aves', 15764),
    ('Procnias nudicollis', 'batbel1', 'Aves', 8854),
    ('Ara ararauna', 'baymac', 'Aves', 19018),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves', 6893),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves', 558564),
    ('Donacobius atricapilla', 'bkcdon', 'Aves', 116877),
    ('Aratinga nenday', 'bkhpar', 'Aves', 367562),
    ('Busarellus nigricollis', 'blchaw1', 'Aves', 5346),
    ('Spizaetus tyrannus', 'blheag1', 'Aves', 5291),
    ('Tityra cayana', 'blttit1', 'Aves', 8830),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves', 16016),
    ('Megarynchus pitangua', 'bobfly1', 'Aves', 16737),
    ('Progne tapera', 'brcmar1', 'Aves', 11870),
    ('Tyto furcata', 'brnowl', 'Aves', 1578502),
    ('Momotus momota', 'bucmot4', 'Aves', 204447),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves', 367564),
    ('Amazona aestiva', 'bufpar', 'Aves', 18978),
    ('Theristicus caudatus', 'bunibi1', 'Aves', 3766),
    ('Athene cunicularia', 'burowl', 'Aves', 19975),
    ('Colaptes campestris', 'camfli1', 'Aves', 18262),
    ('Ortalis canicollis', 'chacha1', 'Aves', 2088),
    ('Mimus saturninus', 'chbmoc1', 'Aves', 14878),
    ('Gnorimopsar chopi', 'chobla1', 'Aves', 10723),
    ('Conirostrum speciosum', 'chvcon1', 'Aves', 10013),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves', 10921),
    ('Micrastur semitorquatus', 'coffal1', 'Aves', 4698),
    ('Nyctidromus albicollis', 'compau', 'Aves', 19627),
    ('Nyctibius griseus', 'compot1', 'Aves', 19667),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves', 12710),
    ('Pachyramphus validus', 'crebec1', 'Aves', 8681),
    ('Taoniscus nanus', 'dwatin1', 'Aves', 20714),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves', 72954),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves', 17231),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves', 144880),
    ('Glaucidium brasilianum', 'fepowl', 'Aves', 19822),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves', 14344),
    ('Myiothlypis flaveola', 'flawar1', 'Aves', 201223),
    ('Tyrannus savana', 'fotfly', 'Aves', 16793),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves', 17076),
    ('Hylocharis chrysura', 'gilhum1', 'Aves', 5979),
    ('Aramides ypecaha', 'giwrai1', 'Aves', 460),
    ('Chionomesa fimbriata', 'glteme1', 'Aves', 1289639),
    ('Saltator coerulescens', 'grasal3', 'Aves', 9850),
    ('Crotophaga major', 'greani1', 'Aves', 1970),
    ('Taraba major', 'greant1', 'Aves', 15957),
    ('Myiopagis viridicata', 'greela', 'Aves', 16892),
    ('Pitangus sulphuratus', 'grekis', 'Aves', 16956),
    ('Nyctibius grandis', 'grepot1', 'Aves', 19680),
    ('Phacellodomus ruber', 'gretho2', 'Aves', 11632),
    ('Tringa melanoleuca', 'greyel', 'Aves', 3892),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves', 3302),
    ('Eucometis penicillata', 'grhtan1', 'Aves', 10698),
    ('Aramides cajaneus', 'gycwor1', 'Aves', 513889),
    ('Anhima cornuta', 'horscr1', 'Aves', 6908),
    ('Passer domesticus', 'houspa', 'Aves', 13858),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves', 18938),
    ('Elaenia spectabilis', 'larela1', 'Aves', 16734),
    ('Elaenia chiriquensis', 'lesela1', 'Aves', 578460),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves', 10555),
    ('Aramus guarauna', 'limpki', 'Aves', 7),
    ('Dryocopus lineatus', 'linwoo1', 'Aves', 17858),
    ('Coccycua minuta', 'litcuc2', 'Aves', 72740),
    ('Setopagis parvula', 'litnig1', 'Aves', 367507),
    ('Pyrrhura frontalis', 'mabpar', 'Aves', 19162),
    ('Cercomacra melanaria', 'magant1', 'Aves', 15737),
    ('Cissopis leverianus', 'magtan2', 'Aves', 72727),
    ('Polioptila dumicola', 'masgna1', 'Aves', 7509),
    ('Chordeiles nacunda', 'nacnig1', 'Aves', 19661),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves', 1506288),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves', 11511),
    ('Icterus croconotus', 'orbtro3', 'Aves', 62564),
    ('Amazona amazonica', 'orwpar', 'Aves', 18982),
    ('Pandion haliaetus', 'osprey', 'Aves', 116999),
    ('Synallaxis albescens', 'pabspi1', 'Aves', 10999),
    ('Furnarius leucopus', 'palhor3', 'Aves', 11281),
    ('Thraupis palmarum', 'paltan1', 'Aves', 10297),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves', 1982),
    ('Patagioenas picazuro', 'picpig2', 'Aves', 3102),
    ('Legatus leucophaius', 'pirfly1', 'Aves', 17312),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves', 73493),
    ('Inezia inornata', 'platyr1', 'Aves', 16344),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves', 8484),
    ('Theristicus caerulescens', 'pluibi1', 'Aves', 3768),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves', 8483),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves', 16273),
    ('Ara chloropterus', 'ragmac1', 'Aves', 19016),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves', 11201),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves', 10310),
    ('Gallus gallus', 'redjun', 'Aves', 882),
    ('Cariama cristata', 'relser1', 'Aves', 14),
    ('Megaceryle torquata', 'rinkin1', 'Aves', 2552),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves', 145267),
    ('Rupornis magnirostris', 'roahaw', 'Aves', 201041),
    ('Turdus rufiventris', 'rubthr1', 'Aves', 12738),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves', 11718),
    ('Casiornis rufus', 'rufcas2', 'Aves', 17102),
    ('Conopophaga lineata', 'rufgna3', 'Aves', 578313),
    ('Furnarius rufus', 'rufhor2', 'Aves', 11275),
    ('Antrostomus rufus', 'rufnig1', 'Aves', 201066),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves', 11624),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves', 17026),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves', 16833),
    ('Tigrisoma lineatum', 'ruther1', 'Aves', 5048),
    ('Galbula ruficauda', 'rutjac1', 'Aves', 1468),
    ('Arremon flavirostris', 'sabspa1', 'Aves', 10064),
    ('Sicalis flaveola', 'saffin', 'Aves', 9864),
    ('Thraupis sayaca', 'saytan1', 'Aves', 10293),
    ('Columbina squammata', 'scadov1', 'Aves', 3564),
    ('Pionus maximiliani', 'schpar1', 'Aves', 19094),
    ('Phaethornis eurynome', 'scther1', 'Aves', 5622),
    ('Myiarchus ferox', 'shcfly1', 'Aves', 16006),
    ('Accipiter striatus', 'shshaw', 'Aves', 5097),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves', 19645),
    ('Ramphocelus carbo', 'sibtan2', 'Aves', 10056),
    ('Crotophaga ani', 'smbani', 'Aves', 1971),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves', 20570),
    ('Cacicus solitarius', 'sobcac1', 'Aves', 10365),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves', 16972),
    ('Myiozetetes similis', 'socfly1', 'Aves', 16842),
    ('Synallaxis frontalis', 'sofspi1', 'Aves', 10996),
    ('Corythopis delalandi', 'souant1', 'Aves', 17264),
    ('Vanellus chilensis', 'soulap1', 'Aves', 4867),
    ('Chauna torquata', 'souscr1', 'Aves', 6910),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves', 15959),
    ('Synallaxis spixi', 'spispi1', 'Aves', 10915),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves', 1584760),
    ('Piaya cayana', 'squcuc1', 'Aves', 1758),
    ('Dendroplex picus', 'stbwoo2', 'Aves', 72806),
    ('Tapera naevia', 'strcuc1', 'Aves', 1989),
    ('Butorides striata', 'strher2', 'Aves', 62528),
    ('Asio clamator', 'strowl1', 'Aves', 558468),
    ('Eupetomena macroura', 'swthum1', 'Aves', 6065),
    ('Chiroxiphia caudata', 'swtman1', 'Aves', 14306),
    ('Crypturellus tataupa', 'tattin1', 'Aves', 20587),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves', 7480),
    ('Ramphastos toco', 'toctou1', 'Aves', 18793),
    ('Tyrannus melancholicus', 'trokin', 'Aves', 16787),
    ('Megascops choliba', 'trsowl', 'Aves', 19788),
    ('Crypturellus undulatus', 'undtin1', 'Aves', 20592),
    ('Thamnophilus caerulescens', 'varant1', 'Aves', 15757),
    ('Jacana jacana', 'watjac1', 'Aves', 4580),
    ('Pyriglena maura', 'wesfie1', 'Aves', 1286886),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves', 6898),
    ('Biatas nigropectus', 'whbant2', 'Aves', 15902),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves', 201224),
    ('Melanerpes candidus', 'whiwoo1', 'Aves', 18183),
    ('Synallaxis albilora', 'whlspi1', 'Aves', 10992),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves', 8469),
    ('Leptotila verreauxi', 'whtdov', 'Aves', 3280),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves', 17786),
    ('Caracara plancus', 'y00678', 'Aves', 4715),
    ('Paroaria capitata', 'yebcar', 'Aves', 10257),
    ('Elaenia flavogaster', 'yebela1', 'Aves', 16714),
    ('Primolius auricollis', 'yecmac', 'Aves', 73272),
    ('Brotogeris chiriri', 'yecpar', 'Aves', 19215),
    ('Daptrius chimachima', 'yehcar1', 'Aves', 1432779),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves', 16567),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name", "inat_taxon_id"])
SCI_LC_TO_LABEL = {str(s).lower(): l for s, l in zip(species_df['scientific_name'], species_df['primary_label'])}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
LABEL_SET = set(species_df['primary_label'])
SCI_LC_SET = set(SCI_LC_TO_LABEL.keys())

# inat_taxon_id mapping (string keys for dir name match)
INAT_ID_TO_LABEL = {}
for _, r in species_df.iterrows():
    iid = r["inat_taxon_id"]
    if iid is not None and iid is not pd.NA and not pd.isna(iid):
        INAT_ID_TO_LABEL[str(int(iid))] = r["primary_label"]
print(f"BC2026 species: {len(species_df)}")
print(f"  by class: {species_df['class_name'].value_counts().to_dict()}")
print(f"  inat_taxon_id mapping: {len(INAT_ID_TO_LABEL)} (sample: {list(INAT_ID_TO_LABEL.keys())[:5]})")


In [ ]:
# ============================================================
# Cell 3: Helpers
# ============================================================
def safe_dir(name):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(name))

def check_storage_gb():
    try:
        return sum(f.stat().st_size for f in AUDIO_DIR.rglob("*") if f.is_file()) / 1e9
    except Exception:
        return 0.0

def safety_ok():
    gb = check_storage_gb()
    if gb > STORAGE_CAP_GB:
        print(f"  ⚠ storage cap reached ({gb:.2f} GB), stopping")
        return False
    return True

all_metadata = []
print("OK helpers defined")


In [ ]:
# ============================================================
# Cell 4: Generic extract function (handles all sources uniformly)
# ============================================================
def extract_source(label, source_id, df_meta, audio_root, name_col, label_col=None, sci_col=None,
                   max_per_species=MAX_PER_SPECIES):
    """Generic extract: match df_meta to BC2026 species, copy audio.

    df_meta: source metadata DataFrame
    audio_root: where audio files live
    name_col: column with filename / relative path
    label_col: column with primary_label (e.g., ebird code)
    sci_col: column with scientific_name (fallback)

    Returns: number of files copied
    """
    print(f"\n{'='*60}\n[{label}] extract from {source_id}\n{'='*60}")
    df = df_meta.copy()
    print(f"  source rows: {len(df)}")

    # Match
    if label_col and label_col in df.columns:
        df["_match_label"] = df[label_col].astype(str)
        df["_in_bc26"] = df["_match_label"].isin(LABEL_SET)
        matched_via = f"label_col={label_col}"
    elif sci_col and sci_col in df.columns:
        df["_match_sci"] = df[sci_col].astype(str).str.lower()
        df["_in_bc26"] = df["_match_sci"].isin(SCI_LC_SET)
        df["_match_label"] = df["_match_sci"].apply(lambda s: SCI_LC_TO_LABEL.get(s, None))
        matched_via = f"sci_col={sci_col}"
    else:
        print(f"  [SKIP {label}] no matchable column. cols: {df.columns.tolist()[:10]}")
        return 0

    df_match = df[df["_in_bc26"]].reset_index(drop=True)
    print(f"  matched via {matched_via}: {len(df_match)} / {df['_in_bc26'].count()} = {df['_in_bc26'].sum()} match")
    print(f"  unique BC2026 species matched: {df_match['_match_label'].nunique()}")

    if len(df_match) == 0:
        return 0

    # Per-species cap
    before = len(df_match)
    df_match = df_match.groupby("_match_label", group_keys=False).head(max_per_species).reset_index(drop=True)
    if before > len(df_match):
        print(f"  after per-species cap {max_per_species}: {before} -> {len(df_match)}")

    # Copy files
    n_copied = 0
    n_skipped_exists = 0
    n_failed = 0
    for _, r in tqdm(df_match.iterrows(), total=len(df_match), desc=f"  {label} copy"):
        if not safety_ok(): break

        primary_label = r["_match_label"]
        if not primary_label or pd.isna(primary_label): continue
        rel_path = str(r[name_col])
        src = audio_root / rel_path
        if not src.exists():
            # Try basename only
            for c in audio_root.rglob(Path(rel_path).name):
                src = c; break
        if not src.exists():
            n_failed += 1
            continue

        dst_dir = AUDIO_DIR / source_id / safe_dir(primary_label)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        if dst.exists() and dst.stat().st_size > 100:
            n_skipped_exists += 1
            continue

        try:
            shutil.copy2(src, dst)
            n_copied += 1
            all_metadata.append({
                "filename": str(dst.relative_to(OUT_DIR)),
                "primary_label": primary_label,
                "scientific_name": str(r.get(sci_col, "") if sci_col else ""),
                "source": source_id,
                "class_name": LABEL_TO_CLASS.get(primary_label, ""),
                "file_size_mb": dst.stat().st_size / 1e6,
            })
        except Exception as e:
            n_failed += 1

    print(f"  [{label}] copied: {n_copied}, skip-exist: {n_skipped_exists}, failed: {n_failed}")
    print(f"  current storage: {check_storage_gb():.2f} GB")
    return n_copied


In [ ]:
# ============================================================
# Cell: iNat extraction (REAL structure: scientific_name in dir name)
# ============================================================
# Structure: train-recordings/train/{NNNNN}_{kingdom}_{phylum}_{class}_{order}_{family}_{genus}_{species}/
# Example: 00000_Animalia_Arthropoda_Insecta_Blattodea_Hodotermitidae_Hodotermes_mossambicus
# → parse last 2 components → "Hodotermes mossambicus" = scientific_name

INAT_CANDIDATES = [
    Path("/kaggle/input/datasets/shadowdude/train-recordings"),
    Path("/kaggle/input/train-recordings"),
]
inat_root = next((p for p in INAT_CANDIDATES if p.exists()), None)

if inat_root is None:
    msg = f"❌ iNat NOT MOUNTED. Tried: {INAT_CANDIDATES}"
    print(msg); raise RuntimeError(msg)
print(f"iNat root: {inat_root}")

# Walk into train/ subdir
train_dir = inat_root / "train"
if not train_dir.exists():
    # Try alternate
    candidates = list(inat_root.iterdir())
    train_dir = next((c for c in candidates if c.is_dir()), None)
    if train_dir is None:
        msg = f"❌ iNat: no top-level dir under {inat_root}"
        print(msg); raise RuntimeError(msg)
    print(f"  Using {train_dir} as train dir")
else:
    print(f"  Using train_dir: {train_dir}")

# List species dirs (have name pattern with underscores)
species_dirs = [d for d in train_dir.iterdir() if d.is_dir()]
print(f"  Species dirs in {train_dir.name}/: {len(species_dirs)}")
print(f"  Sample names: {[d.name[:80] for d in species_dirs[:3]]}")

# Parse dir name to extract scientific_name
def parse_inat_dir(name):
    """00000_Animalia_Arthropoda_Insecta_Blattodea_Hodotermitidae_Hodotermes_mossambicus
       → ('Hodotermes', 'mossambicus') -> 'Hodotermes mossambicus' """
    parts = name.split("_")
    if len(parts) < 3:
        return None
    # Last 2 parts = genus + species
    genus = parts[-2]
    species = parts[-1]
    return f"{genus} {species}".lower()

# Match dir names to BC2026 species
matched_dirs = []
for d in species_dirs:
    sci_lc = parse_inat_dir(d.name)
    if sci_lc and sci_lc in SCI_LC_SET:
        label = SCI_LC_TO_LABEL[sci_lc]
        matched_dirs.append((d, label, sci_lc))

print(f"\n  Match to BC2026 scientific_name: {len(matched_dirs)} dirs")
if matched_dirs:
    for d, lbl, sci in matched_dirs[:10]:
        n_files = sum(1 for _ in d.iterdir() if _.is_file())
        print(f"    {sci:35s} → {lbl}  ({n_files} files)")
else:
    msg = f"❌ iNat: 0 BC2026 species matched"
    print(msg)
    sample_parsed = [(d.name, parse_inat_dir(d.name)) for d in species_dirs[:10]]
    print(f"  Sample parsed: {sample_parsed}")
    print(f"  BC2026 sci_lc sample: {list(SCI_LC_SET)[:10]}")
    raise RuntimeError(msg)

# Extract audio
print(f"\nExtracting iNat (cap {MAX_PER_SPECIES}/species)...")
n_copied = 0
for d, primary_label, sci_lc in tqdm(matched_dirs, desc="iNat"):
    if not safety_ok(): break
    audio_files = []
    for ext in ["*.ogg", "*.wav", "*.mp3", "*.flac"]:
        audio_files.extend(list(d.rglob(ext)))
    audio_files = audio_files[:MAX_PER_SPECIES]
    dst_dir = AUDIO_DIR / "inat" / safe_dir(primary_label)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for src in audio_files:
        if not safety_ok(): break
        dst = dst_dir / src.name
        if dst.exists() and dst.stat().st_size > 100:
            continue
        try:
            shutil.copy2(src, dst)
            n_copied += 1
            all_metadata.append({
                "filename": str(dst.relative_to(OUT_DIR)),
                "primary_label": primary_label,
                "scientific_name": sci_lc,
                "source": "inat",
                "class_name": LABEL_TO_CLASS.get(primary_label, ""),
                "file_size_mb": dst.stat().st_size / 1e6,
            })
        except Exception:
            pass

print(f"\n  iNat copied: {n_copied} files from {len(matched_dirs)} species")
print(f"  storage: {check_storage_gb():.2f} GB")


In [ ]:
# ============================================================
# Cell: AnuraSet extraction (site-based audio, need external annotation)
# ============================================================
# Structure: anuraset-v2-raw/AnuraSet_v2.0.0_raw/{site_id}/{site_id}_{date}_{time}.wav
# → audio is MULTI-SPECIES soundscape per site
# → species labels require external annotation file

ANURA_CANDIDATES = [
    Path("/kaggle/input/datasets/bengtlueers/anuraset-v2-raw"),
    Path("/kaggle/input/anuraset-v2-raw"),
]
anura_root = next((p for p in ANURA_CANDIDATES if p.exists()), None)

if anura_root is None:
    msg = f"❌ AnuraSet NOT MOUNTED. Tried: {ANURA_CANDIDATES}"
    print(msg); raise RuntimeError(msg)
print(f"AnuraSet root: {anura_root}")

# Find data dir (AnuraSet_v2.0.0_raw subdir or similar)
data_dir = None
for cand in [anura_root / "AnuraSet_v2.0.0_raw", anura_root]:
    if cand.exists() and any(c.is_dir() for c in cand.iterdir()):
        data_dir = cand
        break
if data_dir is None:
    msg = f"❌ AnuraSet: no data dir found"
    print(msg); raise RuntimeError(msg)
print(f"  Data dir: {data_dir}")

# List site dirs
site_dirs = [d for d in data_dir.iterdir() if d.is_dir()]
print(f"  Site dirs: {len(site_dirs)}")
print(f"  Sample: {[d.name for d in site_dirs[:10]]}")

# Search for annotation file (CSV/parquet/json) ANYWHERE
print(f"\n=== Searching for annotation files ===")
all_annot_files = []
for ext in ["*.csv", "*.CSV", "*.parquet", "*.json", "*.tsv", "*.xlsx"]:
    matches = list(anura_root.rglob(ext))
    if matches:
        all_annot_files.extend(matches)
        print(f"  {ext}: {len(matches)} found")
        for m in matches[:3]:
            print(f"    {m.relative_to(anura_root)}  ({m.stat().st_size/1024:.1f} KB)")

if not all_annot_files:
    # AnuraSet has no annotation in this Dataset
    # Fall back: AnuraSet is unusable for per-species extraction
    print(f"\n⚠️ AnuraSet: NO annotation files (CSV/parquet/json) found")
    print(f"  AnuraSet structure (site-based soundscape) requires external annotation")
    print(f"  Skipping AnuraSet (no fatal error, continuing without AnuraSet data)")
else:
    # Try to use annotation file
    meta_path = None
    for f in all_annot_files:
        try:
            if f.suffix.lower() == ".csv":
                df_peek = pd.read_csv(f, nrows=5)
            elif f.suffix.lower() == ".parquet":
                df_peek = pd.read_parquet(f).head(5)
            elif f.suffix.lower() == ".json":
                df_peek = pd.read_json(f, lines=True, nrows=5)
            else:
                continue
            cols = df_peek.columns.tolist()
            print(f"  {f.name}: cols={cols[:10]}")

            sci_col = next((c for c in ["scientific_name", "species", "species_name", "label", "taxon"]
                            if c in cols), None)
            name_col = next((c for c in ["filename", "file", "audio", "path", "wav_path", "audio_path", "fname"]
                             if c in cols), None)
            if sci_col and name_col:
                meta_path = f
                print(f"    ★ Found: {sci_col}={sci_col}, name_col={name_col}")
                break
        except Exception as e:
            print(f"  read err: {str(e)[:80]}")

    if meta_path:
        df = pd.read_csv(meta_path) if meta_path.suffix.lower() == ".csv" else pd.read_parquet(meta_path)
        extract_source("AnuraSet", "anuraset", df, anura_root,
                       name_col=name_col, sci_col=sci_col)
    else:
        print(f"\n⚠️ AnuraSet: annotation files found but no usable species+filename column")
        print(f"  Skipping AnuraSet")


In [ ]:
# ============================================================
# Cell 5: Extract BC2021-2025
# ============================================================
# Path candidates per year
def find_bc_root(year):
    candidates = [
        Path(f"/kaggle/input/competitions/birdclef-{year}"),
        Path(f"/kaggle/input/birdclef-{year}"),
    ]
    return next((p for p in candidates if p.exists()), None)

def find_audio_dir(root):
    audio_candidates = [
        root / "train_audio",
        root / "train_short_audio",
    ]
    return next((p for p in audio_candidates if p.exists()), None)

def find_metadata(root):
    meta_candidates = [
        root / "train_metadata.csv",
        root / "train.csv",
        root / "train_short_audio_metadata.csv",
    ]
    return next((p for p in meta_candidates if p.exists()), None)

for year in [2021, 2022, 2023, 2024, 2025]:
    if not safety_ok(): break
    root = find_bc_root(year)
    if root is None:
        print(f"\nBC{year} not mounted (rule accept か input attach 確認)")
        continue
    audio_root = find_audio_dir(root)
    meta_path = find_metadata(root)
    if meta_path is None or audio_root is None:
        print(f"\nBC{year}: meta={meta_path}, audio={audio_root} - skip")
        continue

    df = pd.read_csv(meta_path)
    print(f"\n[BC{year}] meta={meta_path.name} ({len(df)} rows)")
    print(f"  columns: {df.columns.tolist()[:10]}")

    if "filename" not in df.columns:
        print(f"  [skip] no 'filename' column")
        continue

    label_col = "primary_label" if "primary_label" in df.columns else None
    sci_col = "scientific_name" if "scientific_name" in df.columns else None

    extract_source(f"BC{year}", f"bc{year}", df, audio_root,
                   name_col="filename", label_col=label_col, sci_col=sci_col)


In [ ]:
# ============================================================
# Cell 8: Summary + per-source / per-species stats
# ============================================================
if all_metadata:
    final_df = pd.DataFrame(all_metadata)
    print(f"Total extracted: {len(final_df)} files")
    print(f"  by source: {final_df['source'].value_counts().to_dict()}")
    print(f"  by class:  {final_df['class_name'].value_counts().to_dict()}")
    print(f"  unique BC2026 species: {final_df['primary_label'].nunique()} / 234")
    print(f"  total size: {final_df['file_size_mb'].sum()/1024:.2f} GB")
    print(f"  total time: {(time.time()-START_T)/60:.1f} min")

    final_df.to_csv(OUT_DIR / "metadata.csv", index=False)
    print(f"\nSaved: {OUT_DIR / 'metadata.csv'}")

    # Per-source
    src_sum = final_df.groupby("source").agg(
        n_files=("filename", "count"),
        n_species=("primary_label", "nunique"),
        total_mb=("file_size_mb", "sum"),
    ).reset_index()
    src_sum.to_csv(OUT_DIR / "per_source.csv", index=False)
    print(f"\n=== Per-source ===")
    print(src_sum.to_string(index=False))

    # Per-species
    sp_sum = final_df.groupby(["primary_label", "class_name"]).agg(
        n_files=("filename", "count"),
        n_sources=("source", "nunique"),
        total_mb=("file_size_mb", "sum"),
    ).reset_index().sort_values("n_files", ascending=False)
    sp_sum.to_csv(OUT_DIR / "per_species.csv", index=False)
    print(f"\n=== Top 10 species ===")
    print(sp_sum.head(10).to_string(index=False))

    # Missing species
    covered = set(final_df["primary_label"].unique())
    all_labels = set(species_df["primary_label"])
    missing = sorted(all_labels - covered)
    print(f"\n=== Missing species (no data in any source): {len(missing)} ===")
    if missing:
        miss_df = species_df[species_df["primary_label"].isin(missing)]
        print(f"  by class: {miss_df['class_name'].value_counts().to_dict()}")
        miss_df.to_csv(OUT_DIR / "missing_species.csv", index=False)
else:
    print("No metadata accumulated - all sources may have been skipped")


In [ ]:
# ============================================================
# Cell 9: Upload as Kaggle Dataset
# ============================================================
import json
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

USER = "maekeso"
SLUG = "birdclef2026-exp047-bc-inat-anura-extracted"
TITLE = "BirdCLEF2026 exp047 BC+iNat+AnuraSet Extracted"

DRY_RUN = False  # set False to upload

if not DRY_RUN:
    meta = {
        "title": TITLE,
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "other"}],
    }
    (OUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    try:
        api.dataset_create_version(folder=str(OUT_DIR),
                                    version_notes="Phase 2 BC+iNat+AnuraSet extracted",
                                    dir_mode="zip", quiet=False)
        print("OK new version uploaded")
    except Exception:
        try:
            api.dataset_create_new(folder=str(OUT_DIR), public=False,
                                    dir_mode="zip", quiet=False)
            print("OK new dataset created")
        except Exception as e:
            print(f"upload err: {str(e)[:300]}")
    print(f"URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
else:
    print(f"DRY_RUN={DRY_RUN}, skip upload")
